In [45]:
#Install Dependencies
!pip install statsbombpy scikit-learn xgboost lightgbm shap pandas numpy matplotlib seaborn --quiet

In [46]:
#Import Libraries
import pandas as pd
import numpy as np
import warnings
import os

from statsbombpy import sb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

In [47]:
#Configuration
COMPETITIONS = [
    {"competition_id": 43, "season_id": 106, "name": "FIFA World Cup 2022"},
    {"competition_id": 55, "season_id": 282, "name": "UEFA Euro 2024"},
    {"competition_id": 9,  "season_id": 281, "name": "Bundesliga 2023/2024"},
    {"competition_id": 1267, "season_id": 107, "name": "African Cup of Nations 2023"},
    {"competition_id": 53, "season_id": 315, "name": "UEFA Women's Euro 2025"},
    {"competition_id": 7,  "season_id": 235, "name": "Ligue 1 2022/2023"},
    {"competition_id": 43,  "season_id": 3,   "name": "FIFA World Cup 2018"},
    {"competition_id": 72,  "season_id": 107, "name": "Women's World Cup 2023"},
    {"competition_id": 223, "season_id": 282, "name": "Copa America 2024"},
]
WINDOW_MIN   = 15
MIN_POST_MIN = 5
IMPACT_POS_THRESHOLD =  0.10
IMPACT_NEG_THRESHOLD = -0.10

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config set. Window={WINDOW_MIN}min | Thresholds: +{IMPACT_POS_THRESHOLD} / {IMPACT_NEG_THRESHOLD}")

Config set. Window=15min | Thresholds: +0.1 / -0.1


In [48]:
#Helper functions
def get_score_at_minute(events_df, team_name, minute):
    goals = events_df[
        (events_df["type"] == "Shot") &
        (events_df["shot_outcome"] == "Goal") &
        (events_df["minute"] < minute)
    ]
    team_goals = goals[goals["team"] == team_name].shape[0]
    opp_goals  = goals[goals["team"] != team_name].shape[0]
    return team_goals, opp_goals

def get_xg_in_window(events_df, team_name, start_min, end_min):
    window = events_df[
        (events_df["type"] == "Shot") &
        (events_df["minute"] >= start_min) &
        (events_df["minute"] < end_min)
    ].copy()
    team_xg = window[window["team"] == team_name]["shot_statsbomb_xg"].sum()
    opp_xg  = window[window["team"] != team_name]["shot_statsbomb_xg"].sum()
    return round(team_xg, 4), round(opp_xg, 4)

def get_event_counts_in_window(events_df, team_name, start_min, end_min):
    window = events_df[
        (events_df["team"] == team_name) &
        (events_df["minute"] >= start_min) &
        (events_df["minute"] < end_min)
    ]
    shots     = (window["type"] == "Shot").sum()
    passes    = (window["type"] == "Pass").sum()
    pressures = (window["type"] == "Pressure").sum()
    return int(shots), int(passes), int(pressures)

def get_position_group(position_name):
    if not position_name:
        return "Unknown"
    pos = position_name.lower()
    if any(p in pos for p in ["forward", "wing", "striker", "second striker"]):
        return "FW"
    elif any(p in pos for p in ["midfield", "attacking mid", "defensive mid"]):
        return "MF"
    elif any(p in pos for p in ["back", "sweeper", "wing back"]):
        return "DF"
    elif "goalkeeper" in pos or "keeper" in pos:
        return "GK"
    else:
        return "Unknown"

def get_game_phase(minute):
    if minute <= 30:
        return "0-30"
    elif minute <= 60:
        return "31-60"
    elif minute <= 75:
        return "61-75"
    else:
        return "76-90+"

def classify_impact(score):
    if score > IMPACT_POS_THRESHOLD:
        return "Positive"
    elif score < IMPACT_NEG_THRESHOLD:
        return "Negative"
    else:
        return "Neutral"

In [49]:
#Load All Matches
print("\nLoading match lists from StatsBomb...")
all_matches = []

for comp in COMPETITIONS:
    try:
        matches = sb.matches(
            competition_id=comp["competition_id"],
            season_id=comp["season_id"]
        )
        matches["competition_name"] = comp["name"]
        all_matches.append(matches)
        print(f"{comp['name']}: {len(matches)} matches")
    except Exception as e:
        print(f"FAILED: {comp['name']}: {e}")

matches_df = pd.concat(all_matches, ignore_index=True)
print(f"\nTotal matches loaded: {len(matches_df)}")


Loading match lists from StatsBomb...
FIFA World Cup 2022: 64 matches
UEFA Euro 2024: 51 matches
Bundesliga 2023/2024: 34 matches
African Cup of Nations 2023: 52 matches
UEFA Women's Euro 2025: 31 matches
Ligue 1 2022/2023: 32 matches
FIFA World Cup 2018: 64 matches
Women's World Cup 2023: 64 matches
Copa America 2024: 32 matches

Total matches loaded: 424


In [50]:
print("\nExtracting substitution-level features...")
records = []
failed  = 0
match_ids = matches_df["match_id"].tolist()

i = 0
for match_id in match_ids:

    if i % 20 == 0:
        print(f"   Progress: {i}/{len(match_ids)} matches | Samples so far: {len(records)}")
    i += 1

    try:
        events = sb.events(match_id=match_id)

        # Normalize columns
        if "type" in events.columns and events["type"].dtype == object:
            if type(events["type"].iloc[0]) == dict:
                events["type"] = events["type"].apply(lambda x: x.get("name", "") if type(x) == dict else x)

        if "shot_outcome" in events.columns and events["shot_outcome"].dtype == object:
            if type(events["shot_outcome"].dropna().iloc[0]) == dict:
                events["shot_outcome"] = events["shot_outcome"].apply(
                    lambda x: x.get("name", "") if type(x) == dict else x
                )

        if "team" in events.columns and events["team"].dtype == object:
            if type(events["team"].iloc[0]) == dict:
                events["team"] = events["team"].apply(lambda x: x.get("name", "") if type(x) == dict else x)

        if "shot_statsbomb_xg" not in events.columns:
            events["shot_statsbomb_xg"] = 0.0
        events["shot_statsbomb_xg"] = pd.to_numeric(events["shot_statsbomb_xg"], errors="coerce").fillna(0.0)

        subs = events[events["type"] == "Substitution"].copy()
        if subs.empty:
            continue

        match_row  = matches_df[matches_df["match_id"] == match_id].iloc[0]
        home_team  = match_row.get("home_team", {})
        if type(home_team) == dict:
            home_team = home_team.get("home_team_name", "")

        match_minute_max = events["minute"].max()
        sub_seq_tracker = {}

        for _, sub in subs.iterrows():
            sub_min  = int(sub["minute"])
            team     = sub["team"] if type(sub["team"]) == str else sub["team"].get("name", "")

            if not team:
                continue

            time_remaining = 90 - sub_min
            if time_remaining < MIN_POST_MIN:
                continue

            raw_position = sub.get("position", None)
            if type(raw_position) == dict:
                position_name = raw_position.get("name", "")
            elif type(raw_position) == str:
                position_name = raw_position
            else:
                position_name = ""
            position_group = get_position_group(position_name)

            sub_seq_tracker[team] = sub_seq_tracker.get(team, 0) + 1
            sub_sequence = sub_seq_tracker[team]

            pre_start = max(0, sub_min - WINDOW_MIN)
            pre_end   = sub_min

            team_xg_prev, opp_xg_prev = get_xg_in_window(events, team, pre_start, pre_end)
            shots_prev, passes_prev, pressures_prev = get_event_counts_in_window(
                events, team, pre_start, pre_end
            )
            team_goals, opp_goals = get_score_at_minute(events, team, sub_min)
            score_diff   = team_goals - opp_goals
            is_home      = 1 if team == home_team else 0
            xg_diff_prev = round(team_xg_prev - opp_xg_prev, 4)

            post_start = sub_min
            post_end   = min(sub_min + WINDOW_MIN, int(match_minute_max) + 1)

            team_xg_next, opp_xg_next = get_xg_in_window(events, team, post_start, post_end)
            xg_diff_next = round(team_xg_next - opp_xg_next, 4)

            impact_score = round(xg_diff_next - xg_diff_prev, 4)
            impact_label = classify_impact(impact_score)

            records.append({
                "match_id":         match_id,
                "competition":      match_row.get("competition_name", ""),
                "team":             team,
                "sub_minute":       sub_min,
                "score_diff":       score_diff,
                "is_home":          is_home,
                "time_remaining":   time_remaining,
                "sub_sequence":     sub_sequence,
                "game_phase":       get_game_phase(sub_min),
                "position_name":    position_name,
                "position_group":   position_group,
                "team_xg_prev15":   team_xg_prev,
                "opp_xg_prev15":    opp_xg_prev,
                "xg_diff_prev15":   xg_diff_prev,
                "shots_prev15":     shots_prev,
                "passes_prev15":    passes_prev,
                "pressures_prev15": pressures_prev,
                "team_xg_next15":   team_xg_next,
                "opp_xg_next15":    opp_xg_next,
                "xg_diff_next15":   xg_diff_next,
                "impact_score":     impact_score,
                "impact_label":     impact_label,
            })

    except Exception as e:
        failed += 1
        continue

print(f"\nExtraction complete!")
print(f"   Total substitution samples: {len(records)}")
print(f"   Failed matches (skipped):   {failed}")


Extracting substitution-level features...
   Progress: 0/424 matches | Samples so far: 0
   Progress: 20/424 matches | Samples so far: 168
   Progress: 40/424 matches | Samples so far: 310
   Progress: 60/424 matches | Samples so far: 469
   Progress: 80/424 matches | Samples so far: 612
   Progress: 100/424 matches | Samples so far: 772
   Progress: 120/424 matches | Samples so far: 934
   Progress: 140/424 matches | Samples so far: 1077
   Progress: 160/424 matches | Samples so far: 1215
   Progress: 180/424 matches | Samples so far: 1362
   Progress: 200/424 matches | Samples so far: 1517
   Progress: 220/424 matches | Samples so far: 1670
   Progress: 240/424 matches | Samples so far: 1828
   Progress: 260/424 matches | Samples so far: 1972
   Progress: 280/424 matches | Samples so far: 2083
   Progress: 300/424 matches | Samples so far: 2183
   Progress: 320/424 matches | Samples so far: 2287
   Progress: 340/424 matches | Samples so far: 2403
   Progress: 360/424 matches | Sampl

In [51]:
df = pd.DataFrame(records)

print("Dataset Overview:")
print(f"   Shape: {df.shape}")

print("\nImpact Label Distribution:")
print(df["impact_label"].value_counts())

print("\nLabel Percentages:")
print(df["impact_label"].value_counts(normalize=True).round(3) * 100)

print("\nPosition Group Distribution:")
print(df["position_group"].value_counts())

# Encode game phase as number
phase_map = {"0-30": 0, "31-60": 1, "61-75": 2, "76-90+": 3}
df["game_phase_enc"] = df["game_phase"].map(phase_map)

# Encode position group as number
position_map = {"FW": 0, "MF": 1, "DF": 2, "GK": 3, "Unknown": 4}
df["position_group_enc"] = df["position_group"].map(position_map)

# Save to CSV
import os
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/substitutions_phase1_raw.csv", index=False)

print("\nDataset saved to outputs/substitutions_phase1_raw.csv")

Dataset Overview:
   Shape: (2976, 22)

Impact Label Distribution:
impact_label
Positive    1279
Negative    1069
Neutral      628
Name: count, dtype: int64

Label Percentages:
impact_label
Positive   43.0000
Negative   35.9000
Neutral    21.1000
Name: proportion, dtype: float64

Position Group Distribution:
position_group
MF    1276
FW    1268
DF     423
GK       9
Name: count, dtype: int64

Dataset saved to outputs/substitutions_phase1_raw.csv


In [62]:
print("Training baseline models...")

FEATURE_COLS = [
    "score_diff", "is_home", "time_remaining",
    "sub_sequence", "game_phase_enc",
    "position_group_enc",
    "team_xg_prev15", "opp_xg_prev15", "xg_diff_prev15",
    "shots_prev15", "passes_prev15", "pressures_prev15",
]

TARGET_COL = "impact_label"

model_df = df[FEATURE_COLS + [TARGET_COL]].dropna()
print(f"   Samples used for modeling: {len(model_df)}")

X = model_df[FEATURE_COLS].values
y = model_df[TARGET_COL].values

le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"   Label classes: {le.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=42, stratify=y_enc
)
print(f"   Train size: {len(X_train)} | Test size: {len(X_test)}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


def evaluate_model(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Macro F1  : {f1:.4f}")
    print(f"   Macro AUC : {auc:.4f}")
    print("-" * 55)
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return {"model": name, "accuracy": acc, "macro_f1": f1, "macro_auc": auc}

results = []

Training baseline models...
   Samples used for modeling: 2976
   Label classes: ['Negative' 'Neutral' 'Positive']
   Train size: 2380 | Test size: 596


In [63]:
# Baseline 1: Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
lr_prob = lr.predict_proba(X_test_sc)
results.append(evaluate_model("Logistic Regression", y_test, lr_pred, lr_prob))


Training Logistic Regression...
   Accuracy  : 0.5856
   Macro F1  : 0.5778
   Macro AUC : 0.7697
-------------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.68      0.62      0.65       214
     Neutral       0.37      0.63      0.46       126
    Positive       0.74      0.54      0.62       256

    accuracy                           0.59       596
   macro avg       0.60      0.59      0.58       596
weighted avg       0.64      0.59      0.60       596



In [64]:
# Baseline 2: Random Forest
print("\nTraining Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)
results.append(evaluate_model("Random Forest", y_test, rf_pred, rf_prob))


Training Random Forest...
   Accuracy  : 0.6292
   Macro F1  : 0.6197
   Macro AUC : 0.8272
-------------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.72      0.67      0.69       214
     Neutral       0.43      0.65      0.52       126
    Positive       0.73      0.59      0.65       256

    accuracy                           0.63       596
   macro avg       0.63      0.63      0.62       596
weighted avg       0.66      0.63      0.64       596



In [58]:
results_df = pd.DataFrame(results)

print("-" * 55)
print("PHASE 1 RESULTS SUMMARY - Baseline Metrics")
print("-" * 55)
print(results_df.to_string(index=False))

-------------------------------------------------------
PHASE 1 RESULTS SUMMARY - Baseline Metrics
-------------------------------------------------------
              model  accuracy  macro_f1  macro_auc
Logistic Regression    0.5856    0.5778     0.7697
      Random Forest    0.6292    0.6197     0.8272
